# S6E11 | Own Models Only | Honest 5-Fold CV

*Playground Series, Season 6 Episode 11: Predicting Credit Card Fraud (ROC-AUC)*

This notebook trains **six models we wrote ourselves**, stacks them, and checks every step on the same
five folds. It uses no public submission files and no public out-of-fold predictions, and nothing is
chosen on the public leaderboard. It is not the top of the board, and it does not try to be: it is a
pipeline whose cross-validation you can trust, built from parts you can reuse.

**Why bother.** The public leaderboard scores 20% of the test rows, about 60,000 rows with about a
thousand frauds. Section 7 measures how much an AUC on that many rows moves by chance; at the top of
the board the steps are smaller than that. The private leaderboard (the other 80%) decides the ranking,
so here the five-fold CV decides every choice.

**What is in it.**

| Section | What it shows |
| --- | --- |
| 1 Setup | the library (the same code as the [GitHub repo](https://github.com/kabartay/playground-series-s6e11), MIT) |
| 2 Data and metric | class balance, the two scores that carry most of the signal |
| 3 Folds, and the one rule | everything learned from labels stays inside the training fold |
| 4 Three GLMs | target-encoded value keys, generator artefacts, GPT-2 token keys |
| 5 Two shallow boosters | 4-leaf LightGBM, and LightGBM started from the GLM ("GLM boost") |
| 6 RealMLP | a network on value-bucket embeddings |
| 7 The stack | nested logistic regression, and how noisy the public board is |
| 8 Zero-fraud zones | four regions with no fraud, checked with their edges refitted in each fold |
| 9 What did not help | measured in the repo, with numbers |
| 10 Submission | |

Run time on Kaggle CPU: about 2.5 hours, most of it the boosters and the RealMLP.

## 1 · Setup

The models come from one training script, `train_gbdt.py`, with its feature builders in `features.py`
and the data loading and folds in `common.py`. They are written to disk below exactly as they are in
the repo, so every model in this notebook is one command line. `tiktoken` (MIT) gives GPT-2's tokeniser
for one feature family; `pytabkit` (Apache-2.0) provides RealMLP.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken", "pytabkit"], check=True)
import os
os.makedirs("src", exist_ok=True)
os.environ["S6E11_THREADS"] = "4"

In [ ]:
%%writefile src/common.py
"""Shared data loading, folds and feature building for Playground S6E11 (credit card fraud, ROC-AUC)."""

from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

ROOT = Path(__file__).resolve().parents[1]
ON_KAGGLE = Path("/kaggle/input").exists()
DATA = Path(os.environ.get("S6E11_DATA", ROOT / "data"))
RUNS = Path(os.environ.get("S6E11_RUNS", "/kaggle/working/runs" if ON_KAGGLE else ROOT / "runs"))


def _find(name: str, *alts: str) -> Path:
    """Locate a data file locally (data/) or anywhere under /kaggle/input on Kaggle."""
    for n in (name, *alts):
        if (DATA / n).exists():
            return DATA / n
    if ON_KAGGLE:
        for n in (name, *alts):
            hits = sorted(Path("/kaggle/input").rglob(n))
            if hits:
                return hits[0]
    raise FileNotFoundError(name)


TARGET = "is_fraud"
N_FOLDS = 5
SEED = 42

CAT_STR = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
BOOL = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt",
]
NUM = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "time_of_day_hour",
    "day_of_week",
    "merchant_risk_score",
    "prior_disputes",
]
RAW = NUM + CAT_STR + BOOL


def load() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return (train, test, original) with booleans as int8 and string categoricals as pandas category.

    Category levels are the union over all three frames so codes agree everywhere.
    """
    tr = pd.read_csv(_find("train.csv"))
    te = pd.read_csv(_find("test.csv"))
    og = pd.read_csv(_find("original.csv", "credit_card_fraud_2026.csv")).drop(columns=["transaction_id"])
    og.insert(0, "id", -1 - np.arange(len(og)))
    for df in (tr, te, og):
        for c in BOOL:
            df[c] = df[c].astype(str).str.lower().map({"true": 1, "false": 0}).astype("int8")
    for c in CAT_STR:
        levels = sorted(set(tr[c]) | set(te[c]) | set(og[c]))
        for df in (tr, te, og):
            df[c] = pd.Categorical(df[c], categories=levels)
    tr[TARGET] = tr[TARGET].astype(int)
    og[TARGET] = og[TARGET].astype(int)
    return tr, te, og


def folds(y: np.ndarray, n: int = N_FOLDS, seed: int = SEED) -> list[tuple[np.ndarray, np.ndarray]]:
    """Fixed stratified folds over the competition train rows (original rows never enter validation)."""
    return list(StratifiedKFold(n, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def save_run(name: str, oof: np.ndarray, test: np.ndarray, meta: dict) -> Path:
    """Write OOF and test predictions plus metadata to runs/<name>/."""
    d = RUNS / name
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / "oof.npy", oof)
    np.save(d / "test.npy", test)
    (d / "meta.json").write_text(json.dumps(meta, indent=2))
    return d

In [ ]:
%%writefile src/features.py
"""Feature sets. Each builder takes (train, test, original) and returns frames with identical columns.

Fold-safe by construction: nothing here looks at the competition target. Target statistics come only
from the original dataset, which never appears in validation.
"""

from __future__ import annotations

import itertools

import numpy as np
import pandas as pd

from common import BOOL, CAT_STR, NUM, RAW, TARGET


def fe_raw(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """Raw columns only."""
    return [df[RAW].copy() for df in (tr, te, og)]


def fe_orig_te(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """Raw columns plus, for every column, the original dataset's fraud rate and count at that value.

    Unseen values get NaN. Smoothing towards the global rate with weight 20 rows.
    """
    out = fe_raw(tr, te, og)
    prior = og[TARGET].mean()
    for c in RAW:
        key = og[c].astype(str)
        g = og.groupby(key, observed=True)[TARGET].agg(["sum", "count"])
        rate = (g["sum"] + 20 * prior) / (g["count"] + 20)
        for src, dst in zip((tr, te, og), out, strict=True):
            k = src[c].astype(str)
            dst[f"ogte_{c}"] = k.map(rate).astype("float32").values
            dst[f"ogcnt_{c}"] = k.map(g["count"]).astype("float32").values
    return out


def fe_combo(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """fe_orig_te plus simple ratios and a count of risk flags."""
    out = fe_orig_te(tr, te, og)
    for src, dst in zip((tr, te, og), out, strict=True):
        dst["flags"] = src[BOOL].sum(axis=1).astype("int8")
        dst["amt_over_bal"] = (src.amount_usd / (src.account_balance_usd + 1)).astype("float32")
        dst["txn_per_hour"] = (src.txn_count_last_24h / (src.hours_since_last_txn + 0.1)).astype("float32")
        dst["risk_x_vel"] = (src.merchant_risk_score * src.velocity_score).astype("float32")
    return out


def fe_count(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """fe_combo plus frequency of each numeric value over train+test+original (unsupervised)."""
    out = fe_combo(tr, te, og)
    allv = pd.concat([tr[NUM], te[NUM], og[NUM]])
    for c in NUM:
        vc = allv[c].value_counts()
        for src, dst in zip((tr, te, og), out, strict=True):
            dst[f"freq_{c}"] = src[c].map(vc).astype("float32").values
    return out


def fe_ogmodel(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """fe_count plus the prediction of a LightGBM trained on the original dataset only.

    Train and test get a model fitted on all original rows; original rows get 5-fold out-of-fold
    predictions so none sees its own label. The competition target is never read.
    """
    import lightgbm as lgb
    import numpy as np
    from sklearn.model_selection import StratifiedKFold

    out = fe_count(tr, te, og)
    params = dict(
        objective="binary",
        learning_rate=0.03,
        n_estimators=400,
        num_leaves=15,
        min_child_samples=50,
        feature_fraction=0.7,
        verbose=-1,
        num_threads=4,
    )
    y = og[TARGET].to_numpy()
    oof = np.zeros(len(og))
    for i_tr, i_va in StratifiedKFold(5, shuffle=True, random_state=0).split(og, y):
        m = lgb.LGBMClassifier(**params).fit(og[RAW].iloc[i_tr], y[i_tr])
        oof[i_va] = m.predict_proba(og[RAW].iloc[i_va])[:, 1]
    m = lgb.LGBMClassifier(**params).fit(og[RAW], y)
    out[0]["ogmodel"] = m.predict_proba(tr[RAW])[:, 1].astype("float32")
    out[1]["ogmodel"] = m.predict_proba(te[RAW])[:, 1].astype("float32")
    out[2]["ogmodel"] = oof.astype("float32")
    return out


CONT = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]
DIGIT_COLS = [
    "merchant_risk_score",
    "velocity_score",
    "hours_since_last_txn",
    "amount_usd",
    "distance_from_home_km",
    "account_balance_usd",
]
TE_TOP = [
    "cvv_retry_count",
    "ip_country_mismatch",
    "is_foreign_transaction",
    "is_new_merchant",
    "billing_shipping_mismatch",
    "auth_method",
    "used_vpn",
    "is_ai_generated_scam_attempt",
    "merchant_category",
    "txn_count_last_24h",
]


def digits(df: pd.DataFrame) -> pd.DataFrame:
    """First and second decimal digit of continuous columns: a modulo artefact of the synthesiser."""
    import numpy as np

    out = {}
    for c in DIGIT_COLS:
        cents = np.rint(df[c].to_numpy() * 100).astype("int64")
        out[f"d1_{c}"] = ((cents // 10) % 10).astype("int8")
        out[f"d2_{c}"] = (cents % 10).astype("int8")
    return pd.DataFrame(out, index=df.index)


def fe_digits(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """Raw columns plus decimal digits."""
    return [
        pd.concat([r, digits(src)], axis=1) for r, src in zip(fe_raw(tr, te, og), (tr, te, og), strict=True)
    ]


JOINT = [
    ("category_risk", ("merchant_category", "merchant_risk_score")),
    ("foreign_ip", ("is_foreign_transaction", "ip_country_mismatch")),
    ("category_amount", ("merchant_category", "q:amount_usd")),
    ("txn_velocity", ("txn_count_last_24h", "q:velocity_score")),
    ("category_age", ("merchant_category", "customer_age")),
    ("amount_risk", ("q:amount_usd", "q:merchant_risk_score")),
    ("hours_velocity", ("q:hours_since_last_txn", "q:velocity_score")),
    ("age_risk", ("customer_age", "q:merchant_risk_score")),
]
"""Column pairs whose effect on fraud is not additive, as measured and published in megayak's notebook
"S6E11 | Per-Value + Digit TE | Shallow-Tree Stack" (Apache-2.0); "q:" means deciles of a float column."""


def token_frame(df: pd.DataFrame) -> dict[str, np.ndarray]:
    """Text keys of each float column's integer part: its first two GPT-2 tokens and its 3-digit chunks.

    The integer part is written as GPT-2 would see a number in running text (" 1371" for 1371.37), so
    the tokeniser's own digit grouping (" 13", "71") becomes a pair of keys; the 3-digit chunks ("137",
    "1") group the same digits a fixed way. Idea after aphoristicartist and busyaprime's public notebooks;
    written here independently. Tokens are computed once per distinct integer (tiktoken, MIT).
    """
    import tiktoken

    enc = tiktoken.get_encoding("gpt2")
    out: dict[str, np.ndarray] = {}
    for c in DIGIT_COLS:
        ints = pd.Series(np.floor(df[c].to_numpy()).astype("int64")).astype(str)
        uniq = pd.unique(ints)
        toks = {u: [enc.decode([t]) for t in enc.encode(" " + u)] for u in uniq}
        first = {u: t[0] for u, t in toks.items()}
        second = {u: (t[1] if len(t) > 1 else "") for u, t in toks.items()}
        out[f"k_{c}_g2a"] = ints.map(first).to_numpy()
        out[f"k_{c}_g2b"] = ints.map(second).to_numpy()
        out[f"k_{c}_c3a"] = ints.str[:3].to_numpy()
        out[f"k_{c}_c3b"] = ints.str[3:].to_numpy()
    return out


def te_keys(
    tr: pd.DataFrame,
    te: pd.DataFrame,
    og: pd.DataFrame,
    pairs: bool,
    digit_keys: bool = False,
    joint: bool = False,
    token_keys: bool = False,
) -> list[pd.DataFrame]:
    """Integer category keys for in-fold target encoding (train, test, original), built on the union of rows.

    Every column's exact value is a key (when it has at most half as many levels as rows); continuous
    columns are also keyed on rounded values; decimal endings are keys; with `pairs`, every pair among
    TE_TOP is a key; with `joint`, every pair in JOINT is a key named j_<name>; with `token_keys`, the first
    two GPT-2 tokens and the 3-digit chunks of each float column's integer part are keys (`token_frame`).
    The encoding itself happens inside each fold in train_gbdt.py.
    """
    import numpy as np

    both = pd.concat([tr[RAW], te[RAW], og[RAW]], ignore_index=True)
    keys = {}
    for c in RAW:
        codes, uniq = pd.factorize(both[c].astype(str))
        if len(uniq) <= len(both) // 2:
            keys[f"k_{c}"] = codes
    for c in CONT:
        keys[f"k_{c}_int"] = pd.factorize(np.round(both[c]).astype("int64"))[0]
        keys[f"k_{c}_r10"] = pd.factorize(np.round(both[c] / 10).astype("int64"))[0]
    for c in ["merchant_risk_score", "velocity_score"]:
        keys[f"k_{c}_int"] = pd.factorize(np.round(both[c]).astype("int64"))[0]
    for c in DIGIT_COLS:
        keys[f"k_{c}_end"] = np.rint(both[c].to_numpy() * 100).astype("int64") % 100
    if digit_keys:
        # The digit-key family of the six float columns: integer part, tens, value at 0.1 resolution,
        # first one and two significant digits, and the length of the printed value.
        for c in DIGIT_COLS:
            x = both[c].to_numpy()
            text = both[c].astype(str)
            sig = text.str.replace(".", "", regex=False).str.lstrip("0-")
            keys[f"k_{c}_floor"] = pd.factorize(np.floor(x).astype("int64"))[0]
            keys[f"k_{c}_ten"] = pd.factorize(np.floor(x / 10).astype("int64"))[0]
            keys[f"k_{c}_dec"] = pd.factorize(np.floor(x * 10 + 1e-6).astype("int64"))[0]
            keys[f"k_{c}_lead1"] = pd.factorize(sig.str[:1])[0]
            keys[f"k_{c}_lead2"] = pd.factorize(sig.str[:2])[0]
            keys[f"k_{c}_len"] = text.str.len().to_numpy()
    if token_keys:
        for name, col in token_frame(both).items():
            keys[name] = pd.factorize(col)[0]
    if pairs:
        from itertools import combinations

        for a, b in combinations(TE_TOP, 2):
            keys[f"k_{a}__{b}"] = pd.factorize(both[a].astype(str) + "_" + both[b].astype(str))[0]
    if joint:
        for name, cols in JOINT:
            parts = [
                pd.qcut(both[c[2:]], 10, labels=False, duplicates="drop").astype(str)
                if c.startswith("q:")
                else both[c].astype(str)
                for c in cols
            ]
            keys[f"j_{name}"] = pd.factorize(parts[0] + "_" + parts[1])[0]
    k = pd.DataFrame(keys).astype("int32")
    n1, n2 = len(tr), len(tr) + len(te)
    return [
        k.iloc[:n1].reset_index(drop=True),
        k.iloc[n1:n2].reset_index(drop=True),
        k.iloc[n2:].reset_index(drop=True),
    ]


ART_PAIR_COLS = [
    "amount_usd",
    "hours_since_last_txn",
    "distance_from_home_km",
    "account_balance_usd",
    "velocity_score",
    "merchant_risk_score",
]


def fe_art(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """Raw columns plus pooled generator artefacts (unsupervised, no labels used).

    For each numeric column with more than 100 distinct values: log1p of how often the exact value occurs
    in train + test (`art_cnt_*`) and whether it occurs in the original data (`art_inog_*`). For each pair
    of ART_PAIR_COLS: whether the pair of values occurs together in one original row (`art_pair_*`).
    Measured in `research_artefacts.py` (D14): such pooled effects are invisible to per-value encodings.
    """
    out = fe_raw(tr, te, og)
    allv = pd.concat([tr[NUM], te[NUM]])
    for c in NUM:
        if allv[c].nunique() <= 100:
            continue
        vc = allv[c].value_counts()
        ogv = set(og[c].unique())
        for src, dst in zip((tr, te, og), out, strict=True):
            dst[f"art_cnt_{c}"] = np.log1p(src[c].map(vc).fillna(0).to_numpy()).astype("float32")
            dst[f"art_inog_{c}"] = src[c].isin(ogv).to_numpy().astype("int8")
    for a, b in itertools.combinations(ART_PAIR_COLS, 2):
        ok = set(zip(og[a], og[b], strict=True))
        for src, dst in zip((tr, te, og), out, strict=True):
            hit = np.fromiter((p in ok for p in zip(src[a], src[b], strict=True)), bool, len(src))
            dst[f"art_pair_{a}__{b}"] = hit.astype("int8")
    return out


def fe_emb(tr: pd.DataFrame, te: pd.DataFrame, og: pd.DataFrame) -> list[pd.DataFrame]:
    """Raw columns plus categorical copies of every numeric column, for networks that embed categories.

    `emb_<col>`: the integer part of each numeric column as a category (levels pooled over train + test +
    original); for each DIGIT_COLS column also `emb_dec_<col>`, its two decimals as a category, and
    `dec_<col>`, the decimals as a number. Unsupervised. Idea from yekenot's public RealMLP notebook (read as
    text): the network learns an embedding per value bucket instead of seeing target encodings.
    """
    out = fe_raw(tr, te, og)
    srcs = (tr, te, og)
    for c in NUM:
        keys = [np.floor(s[c].to_numpy()).astype("int64") for s in srcs]
        levels = np.unique(np.concatenate(keys))
        for k, dst in zip(keys, out, strict=True):
            dst[f"emb_{c}"] = pd.Categorical(k, categories=levels)
    for c in DIGIT_COLS:
        keys = [np.rint((s[c].to_numpy() % 1) * 100).astype("int64") for s in srcs]
        levels = np.unique(np.concatenate(keys))
        for k, dst in zip(keys, out, strict=True):
            dst[f"emb_dec_{c}"] = pd.Categorical(k, categories=levels)
            dst[f"dec_{c}"] = (k / 100).astype("float32")
    return out


FEATURE_SETS = {
    "raw": fe_raw,
    "art": fe_art,
    "ogte": fe_orig_te,
    "combo": fe_combo,
    "count": fe_count,
    "ogmodel": fe_ogmodel,
    "digits": fe_digits,
    "emb": fe_emb,
}
CATS = CAT_STR

In [ ]:
%%writefile src/train_gbdt.py
"""5-fold CV for LightGBM / XGBoost / CatBoost / GLM / MLP / RealMLP; saves OOF and test to runs/<name>/.

Usage: .venv/bin/python src/train_gbdt.py --model lgb --fe ogte --orig 1 --name lgb_ogte

`--seeds K` trains K models per fold with different random seeds and averages them (same folds);
`--rows N` runs on a stratified subsample of N training rows as a smoke test and saves nothing.
"""

from __future__ import annotations

import argparse
import os
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from common import BOOL, NUM, TARGET, folds, load, save_run
from features import CATS, DIGIT_COLS, FEATURE_SETS, te_keys

THREADS = int(os.environ.get("S6E11_THREADS", "4"))  # shared Mac: keep at 4 or below


def glm_design(X: pd.DataFrame) -> np.ndarray:
    """Logit of every target-encoded column, one-hot categoricals, the 0/1 flags and any art_ features."""
    te_cols = [c for c in X.columns if c.startswith(("te_", "tej_"))]
    q = np.clip(X[te_cols].to_numpy(dtype="float64"), 1e-5, 1 - 1e-5)
    parts = [
        np.log(q / (1 - q)),
        pd.get_dummies(X[[c for c in CATS if c in X.columns]], dtype="float32").to_numpy(),
    ]
    flags = [c for c in BOOL if c in X.columns]
    parts.append(X[flags].to_numpy(dtype="float32"))
    art = [c for c in X.columns if c.startswith("art_")]
    if art:
        parts.append(X[art].to_numpy(dtype="float32"))
    return np.hstack(parts)


def make_design(params: dict, Xtr):
    """Return a function mapping a frame to the GLM design: glm_design plus optional numeric splines.

    params: `splines` (knots per numeric column; 0, the default, adds none). Splines are fitted on
    quantile-transformed numeric columns of the training frame Xtr only.
    """
    from sklearn.preprocessing import QuantileTransformer, SplineTransformer

    knots = int(params.get("splines", 0))
    nums = [c for c in NUM if c in Xtr.columns]
    if knots <= 0 or not nums:
        return glm_design
    qt = QuantileTransformer(n_quantiles=1000, subsample=200_000, random_state=0)
    st = SplineTransformer(n_knots=knots, degree=3)
    st.fit(qt.fit_transform(Xtr[nums].to_numpy(dtype="float64")))

    def design(X) -> np.ndarray:
        return np.hstack([glm_design(X), st.transform(qt.transform(X[nums].to_numpy(dtype="float64")))])

    return design


def fit_glm(params: dict, Xtr, ytr, Xs: list) -> tuple[list[np.ndarray], int]:
    """L2 logistic regression on the GLM design; returns logits for each frame in Xs and the iteration count.

    params: `C` (default 0.01) and `splines` (see make_design).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    design = make_design(params, Xtr)
    Dtr = design(Xtr)
    sc = StandardScaler().fit(Dtr)
    m = LogisticRegression(C=params.get("C", 0.01), max_iter=3000)
    m.fit(sc.transform(Dtr), ytr)
    return [m.decision_function(sc.transform(design(X))) for X in Xs], int(m.n_iter_[0])


def fit_mlp(params: dict, Xtr, ytr, Xs: list, seed: int = 0) -> tuple[list[np.ndarray], int]:
    """Small ReLU network on the standardised GLM design; returns probabilities for each frame in Xs.

    params: `splines` (see make_design), `hidden` (layer sizes, default [64, 32]), `alpha` (L2, default
    1e-3), `lr` (default 1e-3). Early stopping on an internal 10% split of the training rows.
    """
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler

    design = make_design(params, Xtr)
    Dtr = design(Xtr)
    sc = StandardScaler().fit(Dtr)
    m = MLPClassifier(
        hidden_layer_sizes=tuple(params.get("hidden", [64, 32])),
        alpha=params.get("alpha", 1e-3),
        learning_rate_init=params.get("lr", 1e-3),
        batch_size=2048,
        max_iter=params.get("max_iter", 40),
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=seed,
    )
    m.fit(sc.transform(Dtr), ytr)
    return [m.predict_proba(sc.transform(design(X)))[:, 1] for X in Xs], int(m.n_iter_)


def fit_realmlp(params: dict, Xtr, ytr, Xs: list, seed: int = 0) -> tuple[list[np.ndarray], int]:
    """RealMLP (pytabkit's RealMLP_TD_Classifier) on the fold's feature frame; probabilities for each frame.

    pytabkit picks its best epoch (and early-stops, if asked) on a validation set. That set is a stratified
    10% holdout of the training rows, never the validation fold, so the OOF stays honest; the network trains
    on the other 90%. `params` go to RealMLP_TD_Classifier unchanged (n_ens, n_epochs, batch_size, lr,
    hidden_sizes, ...). Categoricals are the pandas category columns. Needs `pip install pytabkit`.
    `params["cls"]` picks another pytabkit classifier with the same interface (e.g. TabM_D_Classifier).
    `params["holdout"]` (default 0.1) sets the holdout share; 0 trains on all training rows and hands
    pytabkit a 5% stratified subset of those same rows as its validation set, so any epoch choice is made
    on training rows (still never the validation fold). Meant for fixed-epoch runs without early stopping.
    """
    import pytabkit
    import torch
    from sklearn.model_selection import train_test_split

    torch.set_num_threads(THREADS)
    params = dict(params)
    share = float(params.pop("holdout", 0.1))
    idx = np.arange(len(ytr))
    if share > 0:
        inn, hold = train_test_split(idx, test_size=share, stratify=ytr, random_state=100 + seed)
    else:
        inn = idx
        hold = train_test_split(idx, test_size=0.05, stratify=ytr, random_state=100 + seed)[1]
    Xtr = Xtr.reset_index(drop=True)
    cls = getattr(pytabkit, params.pop("cls", "RealMLP_TD_Classifier"))
    p = dict(device="cpu", n_threads=THREADS, random_state=seed, verbosity=0)
    p.update(params)
    m = cls(**p)
    m.fit(Xtr.iloc[inn], ytr[inn], Xtr.iloc[hold], ytr[hold])
    return [m.predict_proba(X)[:, 1] for X in Xs], int(p.get("n_epochs", 0))


def fit_predict(
    model: str,
    params: dict,
    Xtr,
    ytr,
    wtr,
    Xva,
    yva,
    Xte,
    init=None,
    seed: int = 0,
    n_iter: int | None = None,
):
    """Train one fold; return (val_pred, test_pred, best_iter).

    Boosters early-stop on (Xva, yva) unless `n_iter` is given, in which case they train exactly that many
    rounds and never see yva (used by `--es inner`, which picks n_iter on a holdout inside the training
    fold). `init` is an optional (train, val, test) tuple of logits that the trees start from (GLM boost);
    predictions are then that logit plus the trees' raw score. `seed` sets the model's random seed.
    """
    if model == "glm":
        (lv, lt), it = fit_glm(params, Xtr, ytr, [Xva, Xte])
        return lv, lt, it
    if model == "mlp":
        (pv, pt), it = fit_mlp(params, Xtr, ytr, [Xva, Xte], seed=seed)
        return pv, pt, it
    if model == "realmlp":
        (pv, pt), it = fit_realmlp(params, Xtr, ytr, [Xva, Xte], seed=seed)
        return pv, pt, it
    if model == "lgb":
        import lightgbm as lgb

        p = dict(
            objective="binary",
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=100,
            feature_fraction=0.5,
            bagging_fraction=0.8,
            bagging_freq=1,
            lambda_l2=5.0,
            cat_smooth=20,
            n_estimators=5000,
            num_threads=THREADS,
            verbose=-1,
            random_state=seed,
        )
        p.update(params)
        if n_iter is not None:
            p["n_estimators"] = n_iter
        m = lgb.LGBMClassifier(**p)
        if n_iter is None:
            extra = {} if init is None else {"init_score": init[0], "eval_init_score": [init[1]]}
            m.fit(
                Xtr,
                ytr,
                sample_weight=wtr,
                eval_set=[(Xva, yva)],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(200, verbose=False)],
                **extra,
            )
            best = m.best_iteration_
        else:
            m.fit(Xtr, ytr, sample_weight=wtr, **({} if init is None else {"init_score": init[0]}))
            best = n_iter
        if init is not None:
            return (
                m.predict(Xva, raw_score=True) + init[1],
                m.predict(Xte, raw_score=True) + init[2],
                best,
            )
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1], best
    if model == "xgb":
        import xgboost as xgb

        p = dict(
            learning_rate=0.03,
            max_depth=6,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.5,
            reg_lambda=5.0,
            n_estimators=5000,
            tree_method="hist",
            enable_categorical=True,
            max_cat_to_onehot=1,
            eval_metric="auc",
            early_stopping_rounds=200,
            n_jobs=THREADS,
            random_state=seed,
        )
        p.update(params)
        if n_iter is not None:
            p.update(n_estimators=n_iter, early_stopping_rounds=None)
        m = xgb.XGBClassifier(**p)
        if n_iter is None:
            extra = {} if init is None else {"base_margin": init[0], "base_margin_eval_set": [init[1]]}
            m.fit(Xtr, ytr, sample_weight=wtr, eval_set=[(Xva, yva)], verbose=False, **extra)
            it = m.best_iteration + 1
        else:
            extra = {} if init is None else {"base_margin": init[0]}
            m.fit(Xtr, ytr, sample_weight=wtr, verbose=False, **extra)
            it = n_iter
        if init is not None:
            return (
                m.predict(Xva, output_margin=True, base_margin=init[1], iteration_range=(0, it)),
                m.predict(Xte, output_margin=True, base_margin=init[2], iteration_range=(0, it)),
                it,
            )
        return (
            m.predict_proba(Xva, iteration_range=(0, it))[:, 1],
            m.predict_proba(Xte, iteration_range=(0, it))[:, 1],
            it,
        )
    if model == "cat":
        from catboost import CatBoostClassifier, Pool

        p = dict(
            learning_rate=0.05,
            depth=6,
            l2_leaf_reg=5,
            iterations=5000,
            eval_metric="AUC",
            od_type="Iter",
            od_wait=200,
            thread_count=THREADS,
            verbose=0,
            random_seed=seed,
        )
        p.update(params)
        if n_iter is not None:
            p["iterations"] = n_iter
            p.pop("od_type")
            p.pop("od_wait")
        cats = [c for c in CATS if c in Xtr.columns]

        def pool(X, y=None, w=None, base=None):
            X = X.copy()
            for c in cats:
                X[c] = X[c].astype(str)
            return Pool(X, y, weight=w, cat_features=cats, baseline=base)

        m = CatBoostClassifier(**p)
        if n_iter is not None:
            m.fit(pool(Xtr, ytr, wtr, None if init is None else init[0]))
            if init is not None:
                raw = "RawFormulaVal"
                return (
                    m.predict(pool(Xva), prediction_type=raw) + init[1],
                    m.predict(pool(Xte), prediction_type=raw) + init[2],
                    n_iter,
                )
            return m.predict_proba(pool(Xva))[:, 1], m.predict_proba(pool(Xte))[:, 1], n_iter
        if init is not None:
            m.fit(pool(Xtr, ytr, wtr, init[0]), eval_set=pool(Xva, yva, base=init[1]), use_best_model=True)
            raw = "RawFormulaVal"
            return (
                m.predict(pool(Xva), prediction_type=raw) + init[1],
                m.predict(pool(Xte), prediction_type=raw) + init[2],
                m.get_best_iteration(),
            )
        m.fit(pool(Xtr, ytr, wtr), eval_set=pool(Xva, yva), use_best_model=True)
        return m.predict_proba(pool(Xva))[:, 1], m.predict_proba(pool(Xte))[:, 1], m.get_best_iteration()
    raise ValueError(model)


HIER_LEVELS = ("ten", "floor", "dec", "")
"""Coarse-to-fine keys of one float column for the hierarchical encoding: tens, integer part, 0.1 steps,
exact value ("" is the exact-value key k_<col>). Each level nests inside the one before it."""


def chain_rates(Kfit: np.ndarray, yfit: np.ndarray, Kapp: np.ndarray, m: float) -> np.ndarray:
    """Fraud rate of each applied row at every level of a key chain, shrunk towards the parent level.

    Kfit, Kapp: integer codes (rows x levels, coarse to fine); level l's rate is
    (frauds + m * parent rate) / (rows + m) over the fit rows sharing the key, starting from the overall
    rate. Returns rates (applied rows x levels).
    """
    r = np.full(len(Kapp), yfit.mean())
    out = np.empty((len(Kapp), Kfit.shape[1]))
    for lvl in range(Kfit.shape[1]):
        size = int(max(Kfit[:, lvl].max(), Kapp[:, lvl].max())) + 1
        s = np.bincount(Kfit[:, lvl], weights=yfit, minlength=size)
        n = np.bincount(Kfit[:, lvl], minlength=size)
        c = Kapp[:, lvl]
        r = (s[c] + m * r) / (n[c] + m)
        out[:, lvl] = r
    return out


def hier_te(Ktr: pd.DataFrame, ytr: np.ndarray, others: list, m: float, cv: int) -> list[pd.DataFrame]:
    """Hierarchical target encoding of the DIGIT_COLS key chains (HIER_LEVELS), fitted on the training rows.

    Training rows are cross-fitted over `cv` inner folds; the other frames use all training rows. Columns
    te_h_<col>_<level>. Idea after busyaprime's public "hier" members; written here independently.
    """
    from sklearn.model_selection import StratifiedKFold

    blocks: list[list[np.ndarray]] = [[] for _ in range(1 + len(others))]
    names = []
    for col in DIGIT_COLS:
        keys = [f"k_{col}_{lv}" if lv else f"k_{col}" for lv in HIER_LEVELS]
        keys = [k for k in keys if k in Ktr.columns]
        if len(keys) < 2:
            continue
        names += [f"te_h_{k[2:]}" for k in keys]
        A = Ktr[keys].to_numpy()
        inner = np.empty((len(A), len(keys)))
        for i_fit, i_app in StratifiedKFold(cv, shuffle=True, random_state=0).split(A, ytr):
            inner[i_app] = chain_rates(A[i_fit], ytr[i_fit], A[i_app], m)
        blocks[0].append(inner)
        for j, K in enumerate(others, start=1):
            blocks[j].append(chain_rates(A, ytr, K[keys].to_numpy(), m))
    return [pd.DataFrame(np.hstack(b).astype("float32"), columns=names) for b in blocks]


def inner_rounds(model: str, params: dict, Xa, ya, wa, init, seed: int, fold: int) -> int:
    """Boosting rounds for one fold, chosen without the validation fold.

    Early-stops on a stratified 10% holdout of the training rows, then scales the best round count by
    1/0.9 for the refit on all training rows. With GLM boost the holdout's starting logits come from the
    fold's GLM, which saw those rows; that can only shift the round count, never the validation fold.
    """
    from sklearn.model_selection import train_test_split

    pos = np.arange(len(ya))
    inn, hold = train_test_split(pos, test_size=0.1, stratify=ya, random_state=100 + fold)
    sub = None if init is None else (init[0][inn], init[0][hold], init[0][hold])
    Xi = Xa.iloc[inn] if hasattr(Xa, "iloc") else Xa[inn]
    Xh = Xa.iloc[hold] if hasattr(Xa, "iloc") else Xa[hold]
    _, _, best = fit_predict(model, params, Xi, ya[inn], wa[inn], Xh, ya[hold], Xh, sub, seed=seed)
    best = int(best) + (1 if model == "cat" else 0)  # CatBoost reports a 0-based index
    return max(1, round(best / 0.9))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, choices=["lgb", "xgb", "cat", "glm", "mlp", "realmlp"])
    ap.add_argument("--fe", default="raw", choices=list(FEATURE_SETS))
    ap.add_argument("--orig", type=float, default=1.0, help="weight of original rows; 0 drops them")
    ap.add_argument("--name", required=True)
    ap.add_argument("--params", default="{}", help="JSON overrides of model params")
    ap.add_argument("--folds", type=int, default=5, help="run only the first k folds (quick checks)")
    ap.add_argument(
        "--te",
        default="none",
        choices=["none", "single", "pairs"],
        help="in-fold target encoding of exact-value keys (single) and also top-10 pairs (pairs)",
    )
    ap.add_argument("--te-smooth", default="auto", help='"auto", m-estimate weight, or list 5,20,100')
    ap.add_argument(
        "--keys",
        default="basic",
        choices=["basic", "full", "tokens"],
        help="full adds the digit-key family; tokens adds GPT-2 tokens and 3-digit chunks on top",
    )
    ap.add_argument(
        "--offset",
        default="none",
        choices=["none", "glm"],
        help="glm: the trees start from an in-fold GLM's logit (GLM boost; lgb, xgb or cat)",
    )
    ap.add_argument(
        "--joint",
        default="none",
        choices=["none", "all", "glm"],
        help="joint keys of non-additive column pairs: for every model (all) or the GLM only (glm)",
    )
    ap.add_argument(
        "--key-counts", action="store_true", help="log count of every key's value over train + test"
    )
    ap.add_argument("--te-cv", type=int, default=5, help="inner cross-fitting folds of the target encoder")
    ap.add_argument(
        "--hier",
        type=float,
        default=0,
        help="m > 0 adds hierarchical encodings of each float column's key chain (needs --keys full)",
    )
    ap.add_argument(
        "--es",
        default="val",
        choices=["val", "inner"],
        help="boosters: early-stop on the validation fold (val, optimistic OOF; runs before D22) or pick the"
        " rounds on a 10%% holdout of the training fold and refit on all of it (inner, honest OOF)",
    )
    ap.add_argument("--seeds", type=int, default=1, help="models per fold with different seeds, averaged")
    ap.add_argument("--seed0", type=int, default=0, help="first seed; seeds run seed0 .. seed0 + seeds - 1")
    ap.add_argument("--rows", type=int, default=0, help="smoke test on a stratified subsample; saves nothing")
    a = ap.parse_args()
    import json

    params = json.loads(a.params)
    glm_params = {"C": params.pop("glm_C", 0.01), "splines": params.pop("glm_splines", 0)}

    tr, te, og = load()
    if a.rows:
        from sklearn.model_selection import train_test_split

        tr = train_test_split(tr, train_size=a.rows, stratify=tr[TARGET], random_state=0)[0]
        tr, te = tr.reset_index(drop=True), te.iloc[:1000].reset_index(drop=True)
    Xtr_all, Xte, Xog = FEATURE_SETS[a.fe](tr, te, og)
    y = tr[TARGET].values
    if a.te != "none":
        from sklearn.preprocessing import TargetEncoder

        Ktr, Kte, Kog = te_keys(
            tr,
            te,
            og,
            pairs=a.te == "pairs",
            digit_keys=a.keys in ("full", "tokens"),
            joint=a.joint != "none",
            token_keys=a.keys == "tokens",
        )
        if a.key_counts:
            # Unsupervised: how many train + test rows share each key's value (no labels involved).
            KK = pd.concat([Ktr, Kte], ignore_index=True)
            for c in Ktr.columns:
                vc = KK[c].value_counts()
                for X, K in ((Xtr_all, Ktr), (Xte, Kte), (Xog, Kog)):
                    X[f"art_kc_{c}"] = np.log1p(K[c].map(vc).fillna(0).to_numpy()).astype("float32")
        print(f"target-encoding {Ktr.shape[1]} keys in each fold", flush=True)

    def add_te(i_tr, i_va):
        """Fit the encoder on the fold's competition training rows only (cross-fitted), apply elsewhere.

        `--te-smooth` may list several m-estimate weights (e.g. 5,20,100): one encoding per weight, the
        first named te_<key>, the others te_<key>_m<weight>.
        """
        if a.te == "none":
            return Xtr_all.iloc[i_tr], Xtr_all.iloc[i_va], Xte, Xog
        frames = [[], [], [], []]
        for j, sm in enumerate(a.te_smooth.split(",")):
            smooth = sm if sm == "auto" else float(sm)
            enc = TargetEncoder(target_type="binary", smooth=smooth, cv=a.te_cv, random_state=0)
            suffix = "" if j == 0 else f"_m{sm}"
            pre = ["tej_" if a.joint == "glm" and c.startswith("j_") else "te_" for c in Ktr.columns]
            cols = [f"{p}{c}{suffix}" for p, c in zip(pre, Ktr.columns, strict=True)]
            A = enc.fit_transform(Ktr.iloc[i_tr], y[i_tr]).astype("float32")
            for k, K in enumerate((None, Ktr.iloc[i_va], Kte, Kog)):
                E = A if K is None else enc.transform(K).astype("float32")
                frames[k].append(pd.DataFrame(E, columns=cols))
        if a.hier > 0:
            H = hier_te(Ktr.iloc[i_tr], y[i_tr], [Ktr.iloc[i_va], Kte, Kog], a.hier, a.te_cv)
            for k, h in enumerate(H):
                frames[k].append(h)
        parts = []
        for X, F in zip((Xtr_all.iloc[i_tr], Xtr_all.iloc[i_va], Xte, Xog), frames, strict=True):
            parts.append(pd.concat([X.reset_index(drop=True), *F], axis=1))
        return tuple(parts)

    oof = np.full(len(tr), np.nan)
    test = np.zeros(len(te))
    scores, iters = [], []
    t0 = time.time()
    fl = folds(y)[: a.folds]
    for k, (i_tr, i_va) in enumerate(fl):
        Xa, Xva, Xte_k, Xog_k = add_te(i_tr, i_va)
        ya = y[i_tr]
        wa = np.ones(len(i_tr))
        if a.orig > 0:
            Xa = pd.concat([Xa, Xog_k], ignore_index=True)
            ya = np.concatenate([ya, og[TARGET].values])
            wa = np.concatenate([wa, np.full(len(og), a.orig)])
        init = None
        if a.offset == "glm":
            if a.orig > 0 or a.te == "none":
                raise SystemExit("--offset glm needs --orig 0 and target encoding")
            init, _ = fit_glm(glm_params, Xa, ya, [Xa, Xva, Xte_k])
        if a.model == "realmlp" and a.orig > 0:
            raise SystemExit("--model realmlp needs --orig 0 (no sample weights)")
        if a.model not in ("glm", "mlp"):
            # tej_ columns (joint keys for the GLM only) feed the GLM offset but not the trees.
            drop = [c for c in Xa.columns if c.startswith("tej_")]
            Xa, Xva, Xte_k = (X.drop(columns=drop) for X in (Xa, Xva, Xte_k))
        n_seeds = 1 if a.model == "glm" else a.seeds
        pv, pt, it = 0.0, 0.0, 0
        for s in range(a.seed0, a.seed0 + n_seeds):
            if a.es == "inner" and a.model in ("lgb", "xgb", "cat"):
                n = inner_rounds(a.model, params, Xa, ya, wa, init, seed=s, fold=k)
                v, t, i = fit_predict(a.model, params, Xa, ya, wa, Xva, None, Xte_k, init, seed=s, n_iter=n)
            else:
                v, t, i = fit_predict(a.model, params, Xa, ya, wa, Xva, y[i_va], Xte_k, init, seed=s)
            pv, pt, it = pv + v / n_seeds, pt + t / n_seeds, it + int(i) // n_seeds
        oof[i_va] = pv
        test += pt / len(fl)
        scores.append(roc_auc_score(y[i_va], pv))
        iters.append(int(it))
        print(f"fold {k} auc {scores[-1]:.5f} iter {it} ({time.time() - t0:.0f}s)", flush=True)
    mask = ~np.isnan(oof)
    cv = roc_auc_score(y[mask], oof[mask])
    print(
        f"{a.name}: CV {cv:.5f}  folds {np.round(scores, 5).tolist()}  "
        f"mean {np.mean(scores):.5f} sd {np.std(scores):.5f}"
    )
    if a.folds == 5 and not a.rows:
        save_run(
            a.name,
            oof,
            test,
            dict(vars(a), cv=cv, folds=scores, iters=iters, seconds=round(time.time() - t0)),
        )


if __name__ == "__main__":
    main()

In [ ]:
import json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
sys.path.insert(0, "src")
from common import RUNS, TARGET, folds, load

tr, te, og = load()
y = tr[TARGET].to_numpy()
FOLDS = folds(y)
print(f"train {tr.shape}, test {te.shape}, original {og.shape}")

## 2 · Data and metric

700k synthetic training rows, generated from the 20,000-row
[Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026)
dataset (CC0). Fraud is rare, so ROC-AUC is decided by how the 1.7% of frauds rank against the rest.

In [ ]:
print(f"fraud rate: train {y.mean():.4f} ({y.sum():,} frauds), original {og[TARGET].mean():.4f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for a, c in zip(ax, ["merchant_risk_score", "velocity_score"]):
    r = tr.groupby(pd.qcut(tr[c], 10, duplicates="drop"), observed=True)[TARGET].mean()
    a.bar(range(len(r)), 100 * r.to_numpy(), color="#3b6ea8")
    a.set_xticks(range(len(r)), [f"{iv.right:.0f}" for iv in r.index], fontsize=8)
    a.set_title(f"fraud rate (%) by decile of {c}"); a.set_xlabel("decile upper edge")
plt.tight_layout(); plt.show()
flags = ["is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch",
         "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
print(pd.DataFrame({f: tr.groupby(f)[TARGET].mean() for f in flags}).T.rename(columns={0: "flag 0", 1: "flag 1"}).round(4))

Fraud rises smoothly with both scores and with every flag. No single column is decisive, so the work is
in how each column's values are encoded. Two facts about the synthetic data shape the features below:

- **Exact values repeat.** The generator copies values from the original rows, so a value's own fraud
  rate (a target encoding of the exact value, its integer part, its decimals) carries signal.
- **Pooled artefacts.** Whether a value occurs in the original data at all, and how often it occurs in
  train + test, also carries signal; per-value encodings cannot see it, because each rare value is
  smoothed towards the base rate on its own. These are unsupervised features (`--fe art`).

## 3 · Folds, and the one rule

`StratifiedKFold(5, shuffle=True, random_state=42)` on `is_fraud`, the split most public out-of-fold
files also use. **Everything learned from labels is learned inside the training fold**:

- target encodings are cross-fitted inside the training fold (5 inner folds), never on the scored fold;
- boosters pick their number of rounds on a 10% holdout of the training fold, then refit on all of it
  (`--es inner`). Early-stopping on the scored fold, which many notebooks do, inflated our boosters' CV
  by 0.00004 to 0.00009 each (measured in the repo);
- the stacker in section 7 is itself cross-validated (nested).

`run()` trains one model with `train_gbdt.py` and prints its fold scores.

In [ ]:
RESULTS = {}

def run(name, args):
    # train one model unless its predictions already exist, then report its fold AUCs
    d = RUNS / name
    if not (d / "oof.npy").exists():
        t0 = time.time()
        cmd = [sys.executable, "train_gbdt.py", "--name", name, *args.split()]
        p = subprocess.run(cmd, cwd="src", capture_output=True, text=True)
        if p.returncode:
            print(p.stdout[-3000:], p.stderr[-3000:]); raise RuntimeError(name)
        print(f"{name}: trained in {(time.time() - t0) / 60:.0f} min")
    m = json.loads((d / "meta.json").read_text())
    RESULTS[name] = dict(cv=m["cv"], folds=m["folds"])
    print(f"{name}: CV {m['cv']:.5f}  folds {np.round(m['folds'], 5).tolist()}")

## 4 · Three GLMs

A logistic regression is a strong model here once each value is encoded well. The design: cubic splines
(8 knots) on every numeric column, one-hot categories, and in-fold target encodings of many **keys**:
each column's exact value, its integer part, its value rounded to tens, its decimal ending, and (with
`--keys full`) its leading digits, its value at 0.1 resolution and the length of its printed value.

**4a. Raw columns and the value keys** (`--fe raw --keys full`, one smoothing weight).

In [ ]:
run("glm_keys", '--model glm --fe raw --te single --keys full --te-smooth 20 --orig 0 --params {"splines":8}')

**4b. Add the pooled artefacts, three smoothing weights and joint keys.** Each key is encoded with
m-estimate weights 5, 20 and 100 (rare values trusted less or more), and eight column pairs whose effect
is not additive become joint keys (the pairs measured and published in megayak's notebook
*S6E11 | Per-Value + Digit TE | Shallow-Tree Stack*, Apache-2.0).

In [ ]:
run("glm_art", '--model glm --fe art --te single --keys full --te-smooth 5,20,100 --orig 0 --joint glm --params {"splines":8}')

**4c. Add GPT-2 token keys.** The language-model notebooks at the top of this competition read each row
as text. Part of what they see is how GPT-2's tokeniser splits a number: `1371` becomes `" 13"` and
`"71"`. Here the first two tokens of each float column's integer part, and its fixed 3-digit chunks,
become keys for the same target encoding (idea after aphoristicartist's and busyaprime's notebooks;
written independently). The GLM gets part of the language models' view at a fraction of the cost.

In [ ]:
run("glm_tok", '--model glm --fe art --te single --keys tokens --te-smooth 5,20,100 --orig 0 --joint glm --params {"splines":8}')

## 5 · Two shallow boosters

Deep trees overfit the encodings; 4-leaf trees with a low learning rate do best. Both boosters see the
same encoded keys plus the log count of every key's value over train + test (`--key-counts`).

**5a. LightGBM, 4 leaves.**

In [ ]:
LGB = '{"num_leaves":4,"learning_rate":0.03,"feature_fraction":0.5,"min_child_samples":100,"lambda_l2":1.0,"n_estimators":20000}'
run("lgb_l4", f"--model lgb --fe art --te single --keys full --te-smooth 5,20,100 --orig 0 --joint glm --es inner --key-counts --params {LGB}")

**5b. GLM boost.** The same trees, but started from an in-fold GLM's logit (`--offset glm`): the GLM
carries the smooth part, the trees only learn the interactions it misses. This is our best single model.

In [ ]:
run("glm_boost", f"--model lgb --fe art --te single --keys full --te-smooth 5,20,100 --orig 0 --joint glm --es inner --key-counts --offset glm --params {LGB}")

## 6 · RealMLP on value-bucket embeddings

A different view for the stack: no target encodings at all. Every numeric column's integer part and its
two decimals become categories, and RealMLP (pytabkit) learns an embedding per value bucket
(`--fe emb`). Hyperparameters start from yekenot's tuned RealMLP notebook, with 8 ensemble members
instead of 16 for CPU, 3 epochs and no early stopping (the scored fold never picks the model).

In [ ]:
RMLP = ('{"n_ens":8,"n_epochs":3,"batch_size":288,"use_early_stopping":false,"val_metric_name":"1-auc_ovr",'
        '"lr":0.073,"wd":0.038,"sq_mom":0.988,"lr_sched":"flat_anneal","wd_sched":"cos_log_15",'
        '"first_layer_lr_factor":0.25,"embedding_size":5,"max_one_hot_cat_size":16,"hidden_sizes":[512,256,128],'
        '"act":"silu","p_drop":0.05,"p_drop_sched":"pow5","plr_hidden_1":16,"plr_hidden_2":8,"plr_act_name":"gelu",'
        '"plr_lr_factor":0.1151,"plr_sigma":0.75,"ls_eps":0.01,"ls_eps_sched":"sqrt_cos","add_front_scale":false,'
        '"bias_init_mode":"neg-uniform-dynamic-2","tfms":["one_hot","median_center","robust_scale","smooth_clip",'
        '"embedding","l2_normalize"]}')
run("realmlp_emb", f"--model realmlp --fe emb --te none --orig 0 --params {RMLP}")

In [ ]:
table = pd.DataFrame({k: [v["cv"], *v["folds"]] for k, v in RESULTS.items()},
                     index=["CV", "fold 0", "fold 1", "fold 2", "fold 3", "fold 4"]).T
table.round(5)

## 7 · The stack

A logistic regression on the six models' logits. It is **nested**: for each outer fold it is fitted on
the other four folds' out-of-fold predictions and scored on the fifth, so its CV is as honest as the
members'. Regularisation C = 0.1.

In [ ]:
names = list(RESULTS)
def to_logit(oof, test):
    # the GLMs and the GLM boost save logits already (values outside [0, 1]); the others save probabilities
    if oof.min() < 0 or oof.max() > 1:
        return oof, test
    f = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    return f(oof), f(test)
pairs = [to_logit(np.load(RUNS / n / "oof.npy"), np.load(RUNS / n / "test.npy")) for n in names]
Z = np.column_stack([o for o, _ in pairs])
Zt = np.column_stack([t for _, t in pairs])

stack_oof = np.zeros(len(y))
for i_tr, i_va in FOLDS:
    lr = LogisticRegression(C=0.1, max_iter=2000).fit(Z[i_tr], y[i_tr])
    stack_oof[i_va] = lr.decision_function(Z[i_va])
stack_folds = [roc_auc_score(y[i], stack_oof[i]) for _, i in FOLDS]
best = max(RESULTS, key=lambda k: RESULTS[k]["cv"])
print(f"best single ({best}): {RESULTS[best]['cv']:.5f}")
print(f"nested stack:        {roc_auc_score(y, stack_oof):.5f}  folds {np.round(stack_folds, 5).tolist()}")
lr = LogisticRegression(C=0.1, max_iter=2000).fit(Z, y)
stack_test = lr.decision_function(Zt)
print("weights:", dict(zip(names, np.round(lr.coef_[0], 3))))

**How noisy is the public leaderboard?** Draw 60,000 rows (the size of the public part of the test set)
from the stack's out-of-fold predictions many times and look at how much the AUC moves.

In [ ]:
rng = np.random.default_rng(0)
single = np.load(RUNS / best / "oof.npy")
level, diff = [], []
for _ in range(300):
    s = rng.choice(len(y), 60_000, replace=False)
    a = roc_auc_score(y[s], stack_oof[s]); level.append(a)
    diff.append(a - roc_auc_score(y[s], single[s]))
full_gain = roc_auc_score(y, stack_oof) - RESULTS[best]["cv"]
print(f"stack AUC on 60,000 random rows: standard deviation {np.std(level):.5f}")
print(f"stack minus best single, all 700k rows: {full_gain:+.5f}")
print(f"stack minus best single, 60,000 rows:   mean {np.mean(diff):+.5f}, standard deviation {np.std(diff):.5f}, "
      f"below zero in {np.mean(np.array(diff) < 0):.0%} of draws")

Two kinds of noise. The **level** of one file's score moves by the first standard deviation between the
public 20% and the private 80%, so a public score says little about the private one at the third
decimal. The **difference** between two files scored on the same rows moves much less, because both see
the same frauds; but even a real, all-folds gain like the stack's over its best member is sometimes
reversed on 60,000 rows. Steps of 0.00001 to 0.0001, as at the top of the board, are well inside that
noise. Five folds of 700,000 rows are not.

## 8 · Zero-fraud zones

Public notebooks (daoviet, goodpjw2008, Weni Mohamed) found four regions with no fraud among the
training rows:

- zone 1: `merchant_risk_score < 13.1`
- zone 2: `velocity_score <= 4.1` and `merchant_risk_score <= 20`
- zone 3: all eight risk flags clear (`cvv_retry_count` and `prior_disputes` 0, the six binary flags 0),
  `velocity_score <= 4.1` and `merchant_risk_score <= 33.8`
- zone 4: all eight flags clear, `velocity_score <= 6.0` and `merchant_risk_score <= 25`

Sending these rows to the bottom of the ranking helps only if they hold no fraud on unseen rows too.
They were found on all training rows, so scoring them on the same rows would flatter them. Here each
merchant-risk edge is refitted on the four training folds (just below the lowest fraud inside the zone,
never above the published edge) and applied to the held-out fold. The velocity edges stay as published,
a small optimism this check does not remove.

In [ ]:
def clean(d):
    ok = (d["cvv_retry_count"].to_numpy() == 0) & (d["prior_disputes"].to_numpy() == 0)
    for f in flags:
        ok &= d[f].to_numpy().astype(int) == 0
    return ok

ZONES = [  # (condition before the merchant-risk edge, published edge, strict <)
    (lambda d: np.ones(len(d), bool), 13.1, True),
    (lambda d: d["velocity_score"].to_numpy() <= 4.1, 20.0, False),
    (lambda d: clean(d) & (d["velocity_score"].to_numpy() <= 4.1), 33.8, False),
    (lambda d: clean(d) & (d["velocity_score"].to_numpy() <= 6.0), 25.0, False),
]

def zone_rows(d_fit, y_fit, d_app):
    # rows of d_app inside any zone, with each merchant-risk edge refitted on (d_fit, y_fit)
    z = np.zeros(len(d_app), bool)
    m_fit, m_app = d_fit["merchant_risk_score"].to_numpy(), d_app["merchant_risk_score"].to_numpy()
    for cond, pub, strict in ZONES:
        fraud = cond(d_fit) & (y_fit == 1)
        low = m_fit[fraud].min() if fraud.any() else np.inf
        edge, s = (low, True) if low <= pub else (pub, strict)
        z |= cond(d_app) & ((m_app < edge) if s else (m_app <= edge))
    return z

zoned = stack_oof.copy(); n_rows = n_fraud = 0
for i_tr, i_va in FOLDS:
    z = zone_rows(tr.iloc[i_tr], y[i_tr], tr.iloc[i_va])
    n_rows += z.sum(); n_fraud += y[i_va][z].sum()
    zoned[i_va[z]] = stack_oof.min() - 1
zoned_folds = [roc_auc_score(y[i], zoned[i]) for _, i in FOLDS]
print(f"held-out rows in a zone: {n_rows:,}, frauds among them: {n_fraud}")
print(f"stack:         {roc_auc_score(y, stack_oof):.5f}  folds {np.round(stack_folds, 5).tolist()}")
print(f"stack + zones: {roc_auc_score(y, zoned):.5f}  folds {np.round(zoned_folds, 5).tolist()}")
print(f"folds up: {sum(a > b for a, b in zip(zoned_folds, stack_folds))} of 5")

A small gain, but on every fold, and no held-out fraud falls into a refitted zone. The stack already
ranks these rows low; the zones only finish the job.

## 9 · What did not help

Measured in the repo on our full 24-model own stack (nested CV 0.95138), with the rule that a change is
kept only if the stack improves on at least 4 of the 5 folds. Details and every other run:
[docs/DESIGN.md](https://github.com/kabartay/playground-series-s6e11/blob/main/docs/DESIGN.md).

| Idea | Result |
| --- | --- |
| Our own target-first GPT-2 (rows as text), 1 epoch | 0.94396 alone; stack weight about 0, stack 0.95134 |
| The same, 3 epochs on 5 GPUs | 0.94717 alone (+0.003 on every fold); stack 0.95131, 2 of 5 folds up |
| Seed bags of the RealMLPs (3 seeds) | each member +0.0001; stack 0.95139, 1 of 5 folds up |
| RealMLPs trained on all rows instead of a 90% split | members +0.0004; stack 0.95137 |
| A residual LightGBM on raw features, started from the stack | stops after 1 to 7 rounds; 0.95137 |
| The 20,000 original rows added to training | LightGBM -0.0001 (0.94680 against 0.94690) |
| Boosters early-stopped on the scored fold | CV inflated by 0.00004 to 0.00009 per booster |

The pattern: once the stack has a GLM, a GLM boost and a network, a new member has to see something none of
them sees. More of the same, or a stronger version of the same, does not move it.

## 10 · Submission

The stack's test logit, with the zone rows (edges refitted on all training rows) sent to the bottom.

In [ ]:
z_test = zone_rows(tr, y, te)
p = 1 / (1 + np.exp(-stack_test))
p[z_test] *= 1e-6
sub = pd.DataFrame({"id": te["id"], TARGET: p})
sub.to_csv("submission.csv", index=False)
print(f"test rows in a zone: {z_test.sum():,} of {len(te):,}")
sub.head()

## Credits

- Data: the competition, and [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) (CC0).
- Joint pairs (section 4b): megayak, *S6E11 | Per-Value + Digit TE | Shallow-Tree Stack* (Apache-2.0).
- GPT-2 token keys (4c): the idea after aphoristicartist's and busyaprime's notebooks.
- RealMLP settings and value-bucket embeddings (6): yekenot's RealMLP notebook.
- Zero-fraud zones (8): daoviet, goodpjw2008, Weni Mohamed.

All code here is ours, from [github.com/kabartay/playground-series-s6e11](https://github.com/kabartay/playground-series-s6e11) (MIT), which also logs every run, including the
ones that failed. If something here helps you, an upvote is appreciated.